In [8]:
# to configure the llm to load

import os
import dotenv

dotenv.load_dotenv("../.env")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ")


# model = "openai/gpt-oss-120b"
model = "openai/gpt-oss-20b"
# model = "llama-3.3-70b-versatile"
# model = "meta-llama/llama-4-scout-17b-16e-instruct"
# model = "deepseek-r1-distill-llama-70b"

In [9]:
# to load the model from Groq using langchain
# init_chat_model()

from langchain.chat_models import init_chat_model
from langchain_core.rate_limiters import InMemoryRateLimiter

# Define rate limitter(eg: 0.1 request pers second = 6 per minute)
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,
    check_every_n_seconds=0.1, # How often check the clock
    max_bucket_size=10 # allow small burst of requests
)

llm = init_chat_model(
    model=model,
    model_provider="groq",
    temperature=0,
    rate_limiter=rate_limiter
)

# to check the model
response = llm.invoke("What is the capital of France?")
print(response)

content='The capital of France is **Paris**.' additional_kwargs={'reasoning_content': 'The user asks: "What is the capital of France?" The answer: Paris. Provide concise answer.'} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 78, 'total_tokens': 118, 'completion_time': 0.041075728, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.003757183, 'prompt_tokens_details': None, 'queue_time': 0.016837341, 'total_time': 0.044832911}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_fef16ea7fa', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d2c1f-60f2-7691-bad9-a98d95ae1175-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 78, 'output_tokens': 40, 'total_tokens': 118, 'output_token_details': {'reasoning': 22}}


In [20]:
# tools for agents
from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchResults
from langchain_community.utilities import WikipediaAPIWrapper

# a tool to search web
tool_search = DuckDuckGoSearchResults()

# a tool to query wikipedia
wiki_api = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=10000
)
tool_wiki = WikipediaQueryRun(api_wrapper=wiki_api)

tool_set = [tool_search, tool_wiki]

# now need to define manual tool calling
tool_mapping = {
    "duckduckgo_results_json": tool_search,
    "wikipedia": tool_wiki
}


In [21]:
from pprint import pprint
llm_with_tools = llm.bind_tools(tool_set)

response = llm_with_tools.invoke("Who is the president of the United States?")
pprint(response)

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Who is the president of the United States?" As of current date 2026-03-26. The president is Joe Biden? Wait, Joe Biden was president until 2025? Actually Joe Biden was president from 2021 to 2025. In 2025, the next election was in 2024, and the winner was... Let\'s recall: In 2024 US presidential election, the winner was... I think it\'s Joe Biden re-elected? Actually I recall that in 2024, the election was between Joe Biden and Donald Trump. The result was that Joe Biden won re-election. So as of 2026, Joe Biden is still president. But let\'s verify. I should search current events. Use duckduckgo search.', 'tool_calls': [{'id': 'fc_fa6a773b-ab36-4108-ad73-825d885cc845', 'function': {'arguments': '{"query":"current president of the United States 2026"}', 'name': 'duckduckgo_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 194, 'prompt_tokens': 216, 'total_tokens'

In [22]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# to bind the tools to the llm
llm_with_tools = llm.bind_tools(tool_set)

chat_history = []

# system prompt to guide the llm to use the tools effectively
# RESEARCH_SYSTEM_PROMPT = """
# You are an expert researcher. Use tools multiple times to verify facts.
# Identify gaps in information and perform follow-up searches.
# """
RESEARCH_SYSTEM_PROMPT="""
Act as an expert reserch assistent. use tools to come up with fast and accurate solution.
"""

chat_history.append(SystemMessage(content=RESEARCH_SYSTEM_PROMPT))  

User_query = """
why nvidia is not seeing the GPU development by AMD, apple and intel as a threat, however the 
A15 and A16 chips from Tesla as a serious threat?
"""

chat_history.append(HumanMessage(content=User_query))

def call_tool(response):
    print(response.tool_calls[0].get("name"))
    tool_name = response.tool_calls[0].get("name")
    args = response.tool_calls[0].get("args")

    tool = tool_mapping.get(tool_name)
    if tool:
        return tool.invoke(args)
    else:
        raise ValueError(f"Tool '{tool_name}' not found in tool mapping.")
    
while True:
    response = llm_with_tools.invoke(chat_history)
    # need to append the reponse to the chat_history to maintain the context for the llm
    chat_history.append(response)

    if response.tool_calls:
        tool_response = call_tool(response)
        pprint(response.tool_calls)
        chat_history.append(
            # append the tool response with the id to link it to the tool call in the response
            ToolMessage(
                content=str(tool_response), 
                tool_call_id=response.tool_calls[0].get("id")
            )
        )
    else:
        print(response.content)
        break 

**Short answer**

Nvidia’s main competitive moat is its high‑performance GPU architecture that dominates the *training* side of AI and the high‑end gaming/graphics market.  
AMD, Apple and Intel are building GPUs that are great for graphics and for the *inference* workloads that fit into their own SoCs, but they haven’t yet produced a silicon platform that can match Nvidia’s scale, software ecosystem, or the sheer compute‑density that drives the training‑center and data‑center markets.

Tesla’s A15 and A16, on the other hand, are *custom‑ASIC inference engines* that are:

* **Designed from the ground up for automotive‑AI and low‑power inference** – they can deliver the same or better performance per watt than a GPU for the specific workloads that Tesla needs (e.g., perception, path‑planning, sensor fusion).
* **Integrated into Tesla’s own supply chain** – Tesla can ship millions of chips to its vehicles without paying a premium for a third‑party GPU.
* **Scalable to data‑center inferen

In [31]:
chat_history

[HumanMessage(content='Who is the president of the United States?', additional_kwargs={}, response_metadata={}),
 SystemMessage(content='snippet: Thepresidentand vicepresidentoftheUnitedStatesare elected throughtheElectoral College , determined bythenumberofsenators and ..., title: 2028 United States presidential election - Wikipedia, link: https://en.wikipedia.org/wiki/2028_United_States_presidential_election, snippet: Voice: Who Will Really BetheNextPresidentoftheUnitedStates? ... Who Will Really BetheNextPresidentoftheUnitedStates?, title: Who Will Really Be the Next President of the United States?, link: https://foreignpolicy.com/2016/12/27/who-will-really-be-the-next-president-of-the-united-states/, snippet: Bio: A longtime activist whose work centers on race and class, West enteredthepresidential race underthebannerofthePeople’s Party, a third ..., title: Did Joe Biden Drop Out: Who Is Running for President in 2024? |, link: https://www.usnews.com/news/elections/articles/who-is-r